# 03 - Entrenamiento MGCECDL

Este notebook clona el repositorio, carga el journal producido por la busqueda MGCECDL de clasificacion, entrena el modelo con GPU T4 y guarda como checkpoint el modelo con mejor `valid_accuracy`.


In [1]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/jclugor/chec-local-uiti-vano-interpreter.git"
REPO_NAME = "chec-local-uiti-vano-interpreter"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


def ensure_lfs_data(project_root):
    data_path = project_root / "data" / "Indicadores_vano_v3.csv"
    if shutil.which("git-lfs") and (project_root / ".git").exists():
        subprocess.run(["git", "lfs", "install", "--local"], cwd=project_root, check=False)
        subprocess.run(["git", "lfs", "pull"], cwd=project_root, check=True)
    if data_path.exists() and data_path.stat().st_size < 1024:
        head = data_path.read_text(errors="ignore")[:120]
        if "git-lfs" in head:
            raise RuntimeError(
                "Indicadores_vano_v3.csv quedo como puntero Git LFS. "
                "Descarga el archivo LFS antes de continuar."
            )



PROJECT_ROOT = resolve_project_root()
install_project_requirements(PROJECT_ROOT)
ensure_lfs_data(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chec_impacto" or module_name.startswith("chec_impacto."):
        del sys.modules[module_name]

os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OPTUNA_DIR = DATA_DIR / "optuna"

study_paths = {
    "clasificacion": OPTUNA_DIR / "mgcecdl_classification_feature_attention_params.journal",
}

study_paths = {
    modo: path
    for modo, path in study_paths.items()
    if path is not None and Path(path).exists()
}
if not study_paths:
    raise FileNotFoundError(
        "No se encontro el journal MGCECDL de clasificacion en " + str(OPTUNA_DIR)
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OPTUNA_DIR:", OPTUNA_DIR)
print("STUDY_PATHS:", study_paths)
print("MODEL_DIR:", DATA_DIR / "models")
print("GRAPH_DIR:", DATA_DIR / "graphs")
print("CUDA disponible:", torch.cuda.is_available())
print("MPS disponible:", torch.backends.mps.is_available())


Updated Git hooks.
Git LFS initialized.
PROJECT_ROOT: /Users/diego/Documents/CHEC/chec_impacto
DATA_DIR: /Users/diego/Documents/CHEC/chec_impacto/data
OPTUNA_DIR: /Users/diego/Documents/CHEC/chec_impacto/data/optuna
STUDY_PATHS: {'clasificacion': PosixPath('/Users/diego/Documents/CHEC/chec_impacto/data/optuna/mgcecdl_classification_feature_attention_params.journal')}
MODEL_DIR: /Users/diego/Documents/CHEC/chec_impacto/data/models
GRAPH_DIR: /Users/diego/Documents/CHEC/chec_impacto/data/graphs
CUDA disponible: False
MPS disponible: True


In [2]:
from chec_impacto.data import (
    construir_matriz_adyacencia_mgcecdl,
    preparar_splits_estratificados,
    procesar_dataset_completo,
)
from chec_impacto.training import construir_modalidades_mgcecdl, resolve_training_device

VENTANA_CLIMATICA_HORAS = 12
FILTRO_UITI_MAX = None

DATA_PATH_CANDIDATES = [
    DATA_DIR / "Indicadores_vano_v3.csv",
    DATA_DIR / "Indicadores_vano_v2.csv",
    DATA_DIR / "Indicadores_vano_v1.csv",
]
DATASET_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("No se encontro Indicadores_vano_v3/v2/v1.csv en data/.")

VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
DEVICE = resolve_training_device("auto")
print(f"Usando device: {DEVICE}")

datos_procesados = procesar_dataset_completo(
    path_clima=DATASET_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target="UITI_VANO",
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

X = datos_procesados["X"]
y = datos_procesados["y"]
features = datos_procesados["features"]

modality_feature_indices = construir_modalidades_mgcecdl(features)
print(
    "Modos MGCECDL para busqueda/entrenamiento:",
    {name: len(indices) for name, indices in modality_feature_indices.items()},
)

GRAPH_DIR = DATA_DIR / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
graph_files = {
    "matrix": GRAPH_DIR / "mgcecdl_adjacency_matrix.npy",
    "features": GRAPH_DIR / "mgcecdl_feature_order.json",
    "edges": GRAPH_DIR / "mgcecdl_preserved_edges.json",
}
if all(path.exists() for path in graph_files.values()):
    stored_features = json.loads(graph_files["features"].read_text(encoding="utf-8"))
    if list(stored_features) == list(features):
        graph_adjacency_matrix = np.load(graph_files["matrix"])
        graph_preserved_edges = json.loads(graph_files["edges"].read_text(encoding="utf-8"))
        print(f"Grafo MGCECDL cargado desde: {GRAPH_DIR}")
    else:
        print("Grafo existente incompatible con las features actuales. Se reconstruirá en data/graphs.")
        graph_adjacency_matrix, graph_preserved_edges = construir_matriz_adyacencia_mgcecdl(
            features,
            ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
        )
else:
    graph_adjacency_matrix, graph_preserved_edges = construir_matriz_adyacencia_mgcecdl(
        features,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )

if not all(path.exists() for path in graph_files.values()) or list(json.loads(graph_files["features"].read_text(encoding="utf-8"))) != list(features):
    np.save(graph_files["matrix"], graph_adjacency_matrix)
    graph_files["features"].write_text(
        json.dumps(features, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    graph_files["edges"].write_text(
        json.dumps(graph_preserved_edges, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Grafo MGCECDL guardado/actualizado en: {GRAPH_DIR}")
graph_adjacency_tensor = torch.as_tensor(
    graph_adjacency_matrix,
    dtype=torch.float32,
    device=DEVICE,
)
assert graph_adjacency_matrix.shape == (len(features), len(features))
print("Matriz de adyacencia MGCECDL:", graph_adjacency_matrix.shape)
print("Aristas preservadas:", len(graph_preserved_edges))


/Users/diego/Documents/CHEC/chec_impacto/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando device: mps
Cargando datos...
Procesamiento completado.
Shape X: (159470, 70), Shape y: (159470, 1)
Modos MGCECDL para busqueda/entrenamiento: {'climaticos': 50, 'estructurales': 20}
Grafo MGCECDL cargado desde: /Users/diego/Documents/CHEC/chec_impacto/data/graphs
Matriz de adyacencia MGCECDL: (70, 70)
Aristas preservadas: 56


In [3]:
from chec_impacto.models import MGCECDLClassifier
from chec_impacto.training import (
    MGCECDLClassificationLoss,
    calcular_estadisticas_reconstruccion_mgcecdl,
    cargar_estudio_optuna_mgcecdl,
    create_classification_dataloaders,
    escalar_features_minmax_mgcecdl,
    train_mgcecdl_classifier,
)

MAX_EPOCHS = 150
PATIENCE = 200
RANDOM_STATE = 42
WEIGHT_MODALITY_LOSS_BY_RELIABILITY = True
OUTPUT_DIR = DATA_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

splits_por_modo = {
    modo: escalar_features_minmax_mgcecdl(
        preparar_splits_estratificados(
            X,
            y,
            modo=modo,
            random_state=RANDOM_STATE,
        )
    )
    for modo in study_paths
}
print(
    "Rangos X_train MGCECDL escalados:",
    {
        modo: (
            float(np.min(splits["X_train"])),
            float(np.max(splits["X_train"])),
        )
        for modo, splits in splits_por_modo.items()
    },
)
studies = {
    modo: cargar_estudio_optuna_mgcecdl(path, modo)
    for modo, path in study_paths.items()
}
for modo, study in studies.items():
    if not study.trials:
        raise ValueError(f"El Study de {modo} no contiene trials: {study_paths[modo]}")
    print(f"Mejor valor {modo}: {study.best_value}")
    print(f"Mejores parametros {modo}: {study.best_params}")

required_graph_params = {
    "gamma_sup",
    "gamma_agr",
    "gamma_reg",
    "rbf_sigma",
    "lambda_reconstruction",
    "lambda_mutual_information",
}
for modo, study in studies.items():
    missing_params = required_graph_params - set(study.best_params)
    if missing_params:
        raise ValueError(
            f"El Study de {modo} usa un espacio de busqueda anterior y no contiene: "
            f"{sorted(missing_params)}. Ejecuta nuevamente su notebook de busqueda."
        )
from datetime import datetime

from chec_impacto.training.mgcecdl import _build_mgcecdl_optimizer


def _checkpoint_path(base_path: Path) -> Path:
    """Añade timestamp al nombre si el archivo base ya existe."""
    base_path = Path(base_path)
    if base_path.exists():
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        stamped = base_path.parent / f"{base_path.stem}_{ts}{base_path.suffix}"
        print(f"  Archivo existente detectado. Se guardará en: {stamped.name}")
        return stamped
    return base_path


Dataset original: X=(159470, 70), y=(159470, 1)
Splits generados -> Train: (102060, 70), Valid: (25516, 70), Test: (31894, 70)
Modo objetivo: clasificacion

Distribución de clases para estratificación:
Original: [39868 39867 39867 39868]
Train:    [25515 25515 25515 25515]
Valid:    [6379 6379 6379 6379]
Test:     [7974 7973 7973 7974]
Rangos X_train MGCECDL escalados: {'clasificacion': (0.0, 1.0000001192092896)}
Mejor valor clasificacion: 0.7979307101426556
Mejores parametros clasificacion: {'hidden_dim': 192, 'embed_dim': 64, 'dropout': 0.0, 'temperature': 1.700484835260391, 'learning_rate': 0.0014685567719565718, 'weight_decay': 0.0001, 'optimizer_type': 'adamw', 'momentum': 0.5759612338515899, 'batch_size': 1024, 'q': 0.4056386271957924, 'q_d': 0.5583015297771812, 'gamma_sup': 0.010000000000000005, 'gamma_agr': 0.017078243047656073, 'gamma_reg': 0.0282350220583157, 'rbf_sigma': 4.668017104669593, 'lambda_reconstruction': 0.1520965746778718, 'lambda_mutual_information': 0.0523643689

In [4]:
if "clasificacion" in studies:
    classification_params = studies["clasificacion"].best_params
    classification_splits = splits_por_modo["clasificacion"]
    classification_batch_size = int(classification_params["batch_size"])
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    n_classes = int(len(np.unique(classification_splits["y_categories_original"])))
    classification_feature_mean, classification_feature_std = (
        calcular_estadisticas_reconstruccion_mgcecdl(classification_splits["X_train"])
    )

    classification_train_loader, classification_valid_loader = create_classification_dataloaders(
        X_train=classification_splits["X_train"],
        y_train=classification_splits["y_train"],
        X_valid=classification_splits["X_valid"],
        y_valid=classification_splits["y_valid"],
        batch_size=classification_batch_size,
        shuffle_seed=RANDOM_STATE,
    )
    classification_model = MGCECDLClassifier(
        modality_feature_indices=modality_feature_indices,
        n_classes=n_classes,
        hidden_dim=int(classification_params["hidden_dim"]),
        embed_dim=int(classification_params["embed_dim"]),
        dropout=float(classification_params["dropout"]),
        temperature=float(classification_params["temperature"]),
    ).to(DEVICE)
    classification_optimizer = _build_mgcecdl_optimizer(
        model=classification_model,
        optimizer_type=str(classification_params.get("optimizer_type", "adamw")),
        learning_rate=float(classification_params["learning_rate"]),
        momentum=float(classification_params.get("momentum", 0.0)),
        weight_decay=float(classification_params["weight_decay"]),
    )
    classification_loss = MGCECDLClassificationLoss(
        q=float(classification_params["q"]),
        q_d=float(classification_params.get("q_d", 0.5)),
        gamma_sup=float(classification_params["gamma_sup"]),
        gamma_agr=float(classification_params["gamma_agr"]),
        gamma_reg=float(classification_params["gamma_reg"]),
        tau=0.1,
        alpha=0.5,
        weight_modality_loss_by_reliability=WEIGHT_MODALITY_LOSS_BY_RELIABILITY,
        feature_mean=classification_feature_mean,
        feature_std=classification_feature_std,
        adjacency_matrix=graph_adjacency_matrix,
        rbf_sigma=float(classification_params["rbf_sigma"]),
        lambda_reconstruction=float(classification_params["lambda_reconstruction"]),
        lambda_mutual_information=float(classification_params["lambda_mutual_information"]),
    )

    classification_checkpoint_path = _checkpoint_path(OUTPUT_DIR / "mgcecdl_classifier_best.zip")
    print(f"Checkpoint del mejor modelo: {classification_checkpoint_path}")

    classification_result = train_mgcecdl_classifier(
        model=classification_model,
        train_loader=classification_train_loader,
        valid_loader=classification_valid_loader,
        optimizer=classification_optimizer,
        loss_fn=classification_loss,
        device=DEVICE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        checkpoint_path=classification_checkpoint_path,
        verbose=True,
    )

    best_epoch = int(classification_result["best_epoch"])
    best_accuracy = float(classification_result["best_metric"])
    checkpoint_path = Path(classification_result["checkpoint_path"])

    print("\n===== Mejor checkpoint MGCECDL clasificacion =====")
    print(f"Mejor epoch      : {best_epoch}")
    print(f"Mejor accuracy   : {best_accuracy:.6f}")
    print(f"Checkpoint modelo: {checkpoint_path}")

    classification_result


Checkpoint del mejor modelo: /Users/diego/Documents/CHEC/chec_impacto/data/models/mgcecdl_classifier_best.zip
Clasificacion | Epoch 001/150 | train_loss=0.577363 | valid_loss=0.533574 | accuracy=0.548636 | balanced_accuracy=0.548636 | macro_f1=0.543865 | best_accuracy=0.548636@1
Clasificacion | Epoch 002/150 | train_loss=0.509543 | valid_loss=0.491800 | accuracy=0.609853 | balanced_accuracy=0.609853 | macro_f1=0.604748 | best_accuracy=0.609853@2
Clasificacion | Epoch 003/150 | train_loss=0.473561 | valid_loss=0.464849 | accuracy=0.649436 | balanced_accuracy=0.649436 | macro_f1=0.648091 | best_accuracy=0.649436@3
Clasificacion | Epoch 004/150 | train_loss=0.448487 | valid_loss=0.445884 | accuracy=0.670285 | balanced_accuracy=0.670285 | macro_f1=0.662767 | best_accuracy=0.670285@4
Clasificacion | Epoch 005/150 | train_loss=0.430910 | valid_loss=0.435028 | accuracy=0.685531 | balanced_accuracy=0.685531 | macro_f1=0.684023 | best_accuracy=0.685531@5
Clasificacion | Epoch 006/150 | train_lo

In [5]:
model_files = []
for _base_name in ("mgcecdl_classifier_best.zip",):
    _stem = Path(_base_name).stem
    _suffix = Path(_base_name).suffix
    _found = sorted(OUTPUT_DIR.glob(f"{_stem}*{_suffix}"))
    if _found:
        model_files.append(_found[-1])

if len(model_files) != len(studies):
    raise RuntimeError(f"Se esperaban {len(studies)} modelo MGCECDL .zip y se encontraron: {model_files}")

print(f"Modelos guardados en: {OUTPUT_DIR}")
for model_path in model_files:
    print(model_path.name)


Modelos guardados en: /Users/diego/Documents/CHEC/chec_impacto/data/models
mgcecdl_classifier_best.zip
